To Do: multi-label CNN on the 5 lens presentation tags, ResNet50, for review before anything trains.

Decisions taken 2026-09-24:

| | |
|---|---|
| Rows | all 3,377 images that carry at least one LP tag (the 178 double-tagged ones are inside these) |
| Columns | the 5 LP tags only. OOI tags left out, they belong to YOLO |
| Split | 80:20, stratified, seed 42 |
| Augmentation | none |

What changed against a normal single-label CNN, and why:

1. **Classification layer** - was a 1000-way softmax head, now 5 independent outputs.
2. **Loss** - was categorical cross entropy, now binary cross entropy. Softmax forces the
   outputs to sum to 1, so it cannot say "Folded Lens AND Multiple Lenses", which 178 images need.
3. **Target** - was one class index per image, now a float 0/1 vector of length 5.
4. **Metric** - was accuracy, now per-class PR-AUC. At 5.3% prevalence a model that always
   answers "no" scores 94.7% and has learned nothing.
5. **Inference** - was argmax (one winner), now sigmoid per column at 0.5, several can be 1.

Deliberately **not** included, add on request: data augmentation, early stopping, positive
class reweighting, dropout, extra dense layers. The first run is the plain baseline.

In [ ]:
import os
import csv
from pathlib import Path
from collections import Counter

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input

# ============================================================
# Recipe settings. Plain variables, no command line arguments.
# ============================================================

IMG_SIZE = 224          # same input size as every previous CNN run
BATCH    = 8             # same batch the previous CNN runs used (50/8/42)
EPOCHS   = 50           # same as the previous CNN runs (50 / 8 / 42)
LR       = 1e-4
SEED     = 42
THRESHOLD = 0.5         # per-column decision threshold for inference

# Root folder that the image_path column is relative to.
# Set it here, or export FMD_IMAGE_ROOT. Leave empty to check the recipe on random tensors.
IMAGE_ROOT = os.environ.get("FMD_IMAGE_ROOT", "")

LABEL_COLUMNS = [
    "Cavity_Off_Center",
    "Folded_Lens",
    "Multiple_Lenses",
    "Missing_Primary_Package",
    "Missing_Lens",
]
num_classes = len(LABEL_COLUMNS)

tf.keras.utils.set_seed(SEED)
print("tensorflow", tf.__version__)
print("labels:", LABEL_COLUMNS)

1. Dataframe and split

In [ ]:
def find_data_dir():
    """Look for the experiment folder from wherever the notebook was opened."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        for c in (p, p / "experiments" / "multilabel_resnet50"):
            if (c / "final_80_20_dataframe.csv").exists():
                return c
    raise SystemExit("final_80_20_dataframe.csv not found - open the notebook inside the repo")


DATA = find_data_dir()
rows = list(csv.DictReader(open(DATA / "final_80_20_dataframe.csv", newline="", encoding="utf-8")))

print("rows       :", len(rows))
print("split_named:", dict(Counter(r["split_named"] for r in rows)))
print("split      :", dict(Counter(r["split"] for r in rows)))
print()
print("positives per column")
for c in LABEL_COLUMNS:
    n = sum(int(r[c]) for r in rows)
    print("  %-26s %5d  (%.1f%%)" % (c, n, 100 * n / len(rows)))

2. Target: a float 0/1 vector per image, not one class index

An image with two tags gets two 1s. A class index can only hold one value, so the target
has to become a multi-hot vector, and it has to be float32 for the loss.

In [ ]:
def subset(name):
    """image paths and the float multi-hot target for one split."""
    rows_s = [r for r in rows if r["split_named"] == name]
    paths  = [r["image_path"] for r in rows_s]
    y      = np.array([[int(r[c]) for c in LABEL_COLUMNS] for r in rows_s], dtype=np.float32)
    return paths, y


# two-way 80/20 as decided: everything named "train" trains, no validation set
train_paths, y_train = subset("train")
test_paths,  y_test  = subset("test")

print("train", y_train.shape, y_train.dtype)
print("test ", y_test.shape, y_test.dtype)
print()

# the image that carries two tags, which is what makes this multi-label
two = [i for i in range(len(y_train)) if y_train[i].sum() == 2]
print("example with 2 tags:", y_train[two[0]], "->", [c for i, c in enumerate(LABEL_COLUMNS) if y_train[two[0]][i]])
print("train images with 2 tags:", len(two))

assert y_train.dtype == np.float32 and y_train.shape[1] == num_classes
assert set(np.unique(y_train)).issubset({0.0, 1.0})

3. Data pipeline - **no augmentation**

Decode, resize to 224x224, apply the ResNet50 ImageNet preprocessing, batch. Nothing else:
no flips, no rotations, no crops, because a geometric transform can change what the defect is.

The pixels are not in the repo, so the datasets build once `IMAGE_ROOT` points at the drop
folder. Everything above it - dataframe, target, class counts - reads straight from the CSVs.

In [ ]:
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_bmp(img, channels=3)      # every file in this set is a BMP
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))     # 224 x 224
    img = preprocess_input(img)                          # ImageNet scaling for ResNet50
    return img


def make_dataset(paths, y, shuffle):
    """tf.data pipeline over the rows whose files are actually on disk."""
    if shuffle:
        idx = np.random.RandomState(SEED).permutation(len(paths))
        paths, y = [paths[i] for i in idx], y[idx]

    full = [str(Path(IMAGE_ROOT) / p) for p in paths]
    present = [i for i, p in enumerate(full) if Path(p).is_file()]
    if not present:
        return None, 0
    full = [full[i] for i in present]
    y = y[present]

    ds = tf.data.Dataset.from_tensor_slices((full, y))
    ds = ds.map(lambda p, t: (load_image(p), t),
                num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(1000, seed=SEED)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE), len(full)


train_ds, n_train = make_dataset(train_paths, y_train, shuffle=True)

test_ds,  n_test  = make_dataset(test_paths,  y_test,  shuffle=False)  # never shuffled, never trained on

if not (n_train and n_val):
    print("IMAGE_ROOT is empty, or the pixels are not on this machine.")
    print("Point IMAGE_ROOT at the drop folder root and the datasets build from there.")
else:
    print("resolved images: train %d | val %d | test %d" % (n_train, n_val, n_test))

print("augmentation: none")

4. Model - the classification layer is the only part that changes

The ImageNet backbone is kept as it is. Only the head is replaced: one output per LP tag,
**no softmax on the output** - the loss handles the sigmoid, and the raw scores are called logits.

In [ ]:
base = ResNet50(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = True                    # fine-tuning enabled, same as the previous runs

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")
x = base(inputs)
x = layers.GlobalAveragePooling2D()(x)

# ---------------------------------------------------------
# 5 outputs, linear: this replaces the old 1000-way softmax head
# ---------------------------------------------------------
logits = layers.Dense(num_classes, activation=None, name="lp_logits")(x)

model = models.Model(inputs, logits, name="resnet50_multilabel_lp")

print("output shape:", model.output_shape, "(batch, 5 logits)")
print("last layer  :", model.layers[-1].name, model.layers[-1].activation)
print("parameters  : {:,}".format(model.count_params()))

5. Loss - binary cross entropy with logits, not categorical cross entropy

`from_logits=True` means Keras applies the sigmoid internally and numerically safely, so the
model output is fed in **raw**. Never put a sigmoid in front of this loss, that double-applies it.

Why not categorical: softmax normalises the 5 outputs to sum to 1, so Folded Lens and
Multiple Lenses cannot both be on. Binary cross entropy treats each tag as its own yes/no
question, so two tags can be 1 at once.

In [ ]:
loss_fn = tf.keras.losses.BinaryCrossentropy(from_logits=True)
print("loss:", loss_fn.name, "| from_logits =", loss_fn.from_logits)

# sanity check on the one shape that matters: a target with two 1s is valid and finite
example_target = tf.constant([[0.0, 1.0, 1.0, 0.0, 0.0]])
example_logits = tf.constant([[0.0, 2.0, -1.0, 0.0, 0.0]])
print("loss for target [0,1,1,0,0] :", float(loss_fn(example_target, example_logits)))

6. Metrics - per-class PR-AUC (AUPRC)

One AUC per column, each tracking that tag on its own. Their mean is the macro AUPRC, the
single number we quote.

Accuracy is deliberately absent: with prevalences of 5.3% to 42.7% it measures how often the
model says "no", not whether it can tell the classes apart. The **random baseline** column below
is the number every AUPRC has to beat.

In [ ]:
metrics = [
    tf.keras.metrics.AUC(curve="PR", name="auprc_" + c)
    for c in LABEL_COLUMNS
]

print("metrics of record:")
for c, m in zip(LABEL_COLUMNS, metrics):
    print("  ", m.name)
print("  macro = mean of those 5")
print()

prevalence = {c: float(y_train[:, i].mean()) for i, c in enumerate(LABEL_COLUMNS)}
print("random baseline (prevalence on the train split)")
for c, p in prevalence.items():
    print("  %-26s %.4f" % (c, p))

7. What the negative counts are used for

The 0s in the label columns are not extra metadata, they are the labels. Counting them gives the
negatives per tag, and the negatives-to-positives ratio is exactly the **positive weight** used to
reweight the loss so the rare tags are not drowned by the common ones.

This is a lever, not part of the baseline: the table below is printed so the ratio is visible,
and the first run goes out unweighted. Say the word and the weighted run follows.

In [ ]:
print("%-26s %8s %8s %12s" % ("column", "positive", "negative", "pos_weight"))
print("-" * 58)
for i, c in enumerate(LABEL_COLUMNS):
    pos = int(y_train[:, i].sum())
    neg = len(y_train) - pos
    print("%-26s %8d %8d %12.2f" % (c, pos, neg, neg / max(pos, 1)))
print()
print("baseline run: unweighted (pos_weight off)")

8. Compile and train

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=LR, momentum=0.9, nesterov=True),
    loss=loss_fn,
    metrics=metrics,
)

model.summary()

# fixed 50 epochs, no early stopping: the epoch count was chosen before
# this run, so the test set is only touched once, after training ends.
history = model.fit(
    train_ds,
    epochs=EPOCHS,
)

9. Results - every AUPRC read against its own random baseline

The point of the table is the gap column. An AUPRC close to the baseline means the model is
guessing at the right rate, not finding the tag.

In [ ]:
def best_value(key):
    vals = [v for v in history.history.get(key, []) if v is not None]
    return float(np.nanmax(vals)) if vals else float("nan")


print("%-26s %10s %10s %8s" % ("column", "AUPRC", "random", "gap"))
print("-" * 58)
scores = []
for c in LABEL_COLUMNS:
    s = best_value("val_auprc_" + c)
    b = prevalence[c]
    scores.append(s)
    print("%-26s %10.4f %10.4f %+8.4f" % (c, s, b, s - b))
print("-" * 58)
print("%-26s %10.4f" % ("macro AUPRC (mean of 5)", float(np.mean(scores))))
print("%-26s %10.4f" % ("train loss", history.history["loss"][-1]))

10. Precision and recall at 0.5 on the test split

The threshold is per column, so an image can be positive for more than one tag.

In [ ]:
test_logits = model.predict(test_ds, verbose=0)
test_probs  = tf.sigmoid(test_logits).numpy()          # 5 independent probabilities
test_pred   = (test_probs > THRESHOLD).astype(np.int32)  # NOT argmax

print("predicted label counts on the test split (threshold %.2f)" % THRESHOLD)
for i, c in enumerate(LABEL_COLUMNS):
    print("  %-26s %4d" % (c, int(test_pred[:, i].sum())))
print()
print("images predicted positive for more than one tag:", int((test_pred.sum(axis=1) > 1).sum()))
print()

print("%-26s %10s %10s" % ("column", "precision", "recall"))
print("-" * 48)
y_true = y_test[: len(test_pred)]
for i, c in enumerate(LABEL_COLUMNS):
    tp = int(((test_pred[:, i] == 1) & (y_true[:, i] == 1)).sum())
    fp = int(((test_pred[:, i] == 1) & (y_true[:, i] == 0)).sum())
    fn = int(((test_pred[:, i] == 0) & (y_true[:, i] == 1)).sum())
    precision = tp / (tp + fp) if tp + fp else float("nan")
    recall    = tp / (tp + fn) if tp + fn else float("nan")
    print("%-26s %10.3f %10.3f" % (c, precision, recall))

Notes for whoever reads the numbers

- **Folded Lens only ever appears together with Multiple Lenses** (178 of 178 images). A model
  can score on Folded Lens by copying the Multiple Lenses answer without ever looking for a fold.
  Its random baseline is 0.053, and its predictions should be checked by eye before quoting it.
- The test split holds 36 Folded Lens images, so recall there carries roughly plus or minus
  10 points at 90% recall.
- Split is two-way 80/20, seed 42, iterative multi-label stratification (Sechidis 2011):
  2,700 train, 677 test, every tag within 0.88% of its target count, 0 duplicate images
  across splits. No validation set: the epoch count is fixed in advance, so test is
  evaluated once, after training.
- When the completed annotation export arrives, rerun `build_labels.py` and
  `make_split_variants.py`, then rerun this notebook unchanged.